shap

In [1]:
import sys
import os
sys.path.append('..')
from path_config import *

import myfunction as mf
import myrfecv as mr



import pandas as pd
import numpy as np
import xarray as xr
from scipy import stats
from scipy.stats import yeojohnson, boxcox
from lightgbm import LGBMRegressor
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings('ignore')


list_color = ["#ee877c", "#8bd0e3", "#6abeae", "#808eaf", "#f7bba8", "#acb4cc", "#b5e0d5", "#e86462", "#a89687"]

### shap

#### sw


In [2]:
int_rdm = 202603
str_describe = 'sw'
save_file = True

list_pfas = ['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 
             'PFOS', 'FOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', 
             'HFPO-DA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS']

In [3]:
df_sw_raw = pd.read_csv(path_part0_match + "sw_match_geo_auto_zscore.csv")
# df_sw_raw = df_sw_raw.head(8000)
df_sw_data_raw = df_sw_raw.copy()
print(df_sw_data_raw['posname'].unique().tolist())
print(len(df_sw_data_raw['posname'].unique().tolist()))

df_sw_data_raw['value'], lam = boxcox(df_sw_data_raw['value'])
print(f'Box-Cox Lambda: {lam}')



# # MinMax 归一化
# max_val = df_sw_data_raw['value'].max()
# min_val = df_sw_data_raw['value'].min()
# print(f"[MinMax] Max: {max_val}, Min: {min_val}")
# df_sw_data_raw['value'] = (df_sw_data_raw['value'] - min_val) / (max_val - min_val)

mean_val = df_sw_data_raw['value'].mean()
std_val = df_sw_data_raw['value'].std(ddof=0)
print(f"[ZScore] Mean: {mean_val}, Std: {std_val}")
df_sw_data_raw['value'] = (df_sw_data_raw['value'] - mean_val) / std_val
list_inv_parm = [mean_val, std_val, lam]


df_sw_data_raw['year'] = (df_sw_data_raw['year'] - 2000) / (2020 - 2000)

df_sw_train = df_sw_data_raw[df_sw_data_raw['posname'].isin(list_pfas)]
df_sw_tranfer = df_sw_data_raw[~df_sw_data_raw['posname'].isin(list_pfas)]


columns_to_drop = ['lon_grid', 'lat_grid', 'posname']

df_sw_data = df_sw_train.drop(columns=columns_to_drop)
print(df_sw_data.shape)

print(df_sw_data.columns)

# value列范围
print(f"Value range: {df_sw_data['value'].min()} to {df_sw_data['value'].max()}")


['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 'PFOS', 'FOSA', 'EtFOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', '6:2 Cl-PFESA', '6:2 FTSA', '6:2 diPAP', '6:2/8:2 diPAP', '6:8 PFPIA', 'EtFOSAA', 'EtFOSE', 'FOSAA', 'HFPO-DA', 'MeFOSA', 'MeFOSAA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS', 'PFECHS', 'PFHxDA', '4:2 FTSA', '8:2 FTSA', 'PFODA', 'PFPeDA', '8:2 diPAP', '10:2 FTSA']
39
Box-Cox Lambda: 0.008311325940076589
[ZScore] Mean: -1.2075369943593433, Std: 2.9563841404384954
(13256, 43)
Index(['mrro', 'log_Kaw', 'log_pKa', 'pr', 'TOTALS_CO2', 'po_carbon',
       'solubility', 'prsn', 'density', 'log_KHxd_air', 'WWT_CH4', 'po_m_w',
       'tas', 'GDP', 'SWD_LDF_CH4', 'ta', 'urban', 'log_D5_5', 'z', 'zg',
       'log_Koa_wet', 'ps', 'sfcWind', 'distance_to_sources', 'value',
       'fluorite_consumption', 'ts', 'paper_consumption', 'manufacturing',
       'psl', 'wrap_consumption', 'clt', 'log_D7_4', 'potential_contamination',
       'population', 'huss', 'po_chain', '

In [ ]:


selected_features_sw = pd.read_csv(path_part3_sw + str_describe + "_rfecv_features_LGBMcv.csv")
selected_features_sw = selected_features_sw[selected_features_sw["Rank"] == 1]["Feature"].values

best_params = pd.read_csv(path_part3_sw + 'ml_cv_best.csv')

model_params = best_params[best_params["model"] == 'LGBM'].iloc[0]
if pd.isna(model_params["max_depth"]) or str(model_params["max_depth"]).lower() == 'none':
    param_max_depth = None
else:
    param_max_depth = int(float(model_params["max_depth"]))

model_sw = LGBMRegressor(
    max_depth=param_max_depth,
    learning_rate=model_params["learning_rate"],
    min_child_samples=int(model_params["min_child_samples"]),
    num_leaves=int(model_params["num_leaves"]),
    n_estimators=int(model_params["n_estimators"]),
    random_state=20260300,
    subsample=0.8,
    subsample_freq=1,
    n_jobs=12
)


X_sw = df_sw_data[selected_features_sw]
y_sw = df_sw_data['value']


In [5]:
# 跑一次就可以了
import shap
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 确保模型已经训练
model_sw.fit(X_sw, y_sw)

# 创建 explainer 并计算 SHAP 值
try:
    explainer_sw = shap.TreeExplainer(model_sw)
    shap_values_sw = explainer_sw(X_sw)
    # 保存 SHAP 值到本地文件
    shap_save_path_sw = path_part3_fig + 'shap_values_sw.pkl'
    with open(shap_save_path_sw, 'wb') as f:
        pickle.dump(shap_values_sw, f)
    
    print("SHAP values calculated and saved successfully.")
except Exception as e:
    print(f"Error calculating or saving SHAP values: {e}")



SHAP values calculated and saved successfully.


In [11]:
# 读取数据，开始画图
import shap
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
shap_save_path_sw = path_part3_fig + 'shap_values_sw.pkl'
try:
    with open(shap_save_path_sw, 'rb') as f:
        loaded_shap_values_sw = pickle.load(f)
    print("SHAP values loaded successfully.")
except Exception as e:
    print(f"Error loading SHAP values: {e}")
    loaded_shap_values_sw = None


SHAP values loaded successfully.


In [ ]:
Index([
       'paper_consumption', 'manufacturing', 
       'GDP', 'SWD_LDF_CH4',
       'nships_smoothed', 'wrap_consumption', 'clothing',
       'c', 'WWT_CH4', 
       'fluorite_consumption', 'TOTALS_CO2'],
      dtype='object')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# 设置全局字体为 Arial
plt.rcParams['font.family'] = 'Arial'

# 示例：选择一个特征
feature_name = "potential_contamination"
feature_idx = loaded_shap_values_sw.feature_names.index(feature_name)

feature_values = X_sw[feature_name].values
shap_values_for_feature = loaded_shap_values_sw.values[:, feature_idx]

# 分箱计算平均
df_plot = pd.DataFrame({
    'feature_value': feature_values,
    'shap_value': shap_values_for_feature
})
df_plot['bin'] = pd.qcut(df_plot['feature_value'], q=200, duplicates='drop')
bin_means = df_plot.groupby('bin').mean()

# 线性回归拟合
X_param = sm.add_constant(bin_means['feature_value'])  # 截距
y_param = bin_means['shap_value']
model = sm.OLS(y_param, X_param).fit()

alpha = model.params[0]
beta = model.params[1]
formula_text = f"SHAP ≈ {alpha:.3f} + {beta:.3f} × {feature_name}"

# 绘图
plt.figure(figsize=(6,4))
plt.plot(bin_means['feature_value'], bin_means['shap_value'], marker='o', label='平均 SHAP')
plt.plot(bin_means['feature_value'],
         model.predict(X_param),
         color='red', linestyle='--', label='线性拟合')

# 在图上写公式
plt.text(0.05, 0.95, formula_text,
         transform=plt.gca().transAxes,
         fontsize=10,
         fontname='Arial',
         verticalalignment='top',
         bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.3))

plt.axhline(0, color='gray', linewidth=1)
plt.xlabel(feature_name)
plt.ylabel("Average SHAP value")
plt.title(f"SHAP Main Effect: {feature_name}", fontname='Arial')

# 图例放左下角
plt.legend(loc='lower left', prop={'family': 'Arial', 'size': 9})

plt.tight_layout()
plt.show()


shap.dependence_plot(feature_name,
                     shap_values=loaded_shap_values_sw.values,
                     features=X_sw,
                     feature_names=loaded_shap_values_sw.feature_names)


In [22]:
path_part3_fig2 = 'F:/User_file/wyy/SPDB/part3_forecast/sw/fig/'

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
if loaded_shap_values_sw is not None:
    # ===== 全局字体设置 =====
    plt.rcParams.update({
        'font.size': 16,         # 默认字体大小
        'axes.titlesize': 18,    # 标题字体
        'axes.labelsize': 16,    # 坐标轴标签字体
        'xtick.labelsize': 14,   # x轴刻度字体
        'ytick.labelsize': 14,   # y轴刻度字体
        'legend.fontsize': 14    # 图例字体
    })

    # 计算总变量数量
    # X_sw.shape[1]
    num_features = 20

    # =========================
    # Summary plot (bar)
    # =========================
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        loaded_shap_values_sw, 
        X_sw, 
        plot_type="bar",
        show=False, 
        color='#84aeb8',
        max_display=num_features
    )

    ax = plt.gca()
    ax.set_title("SHAP bar plot for water", fontsize=18)

    # x轴label分成两行
    ax.set_xlabel("Mean |SHAP value|\n(average impact on model output magnitude)", fontsize=14)

    # 调整坐标轴刻度字体
    ax.tick_params(axis='both', labelsize=14)

    # 添加数值标签，并放大字体
    for p in ax.patches:
        width = p.get_width()
        ax.text(
            width, 
            p.get_y() + p.get_height() / 2, 
            f'{width:.3f}', 
            ha='left', 
            va='center',
            fontsize=13
        )

    plt.tight_layout()
    plt.savefig(path_part3_fig2 + 'shap_sw_summary_bar_plot.svg', bbox_inches='tight')
    plt.close()




    # =========================
    # Summary plot (dot)
    # =========================
    colors = ['#FFFF00', '#FF0000']
    n_bins = 100
    cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=n_bins)

    fig = plt.figure(figsize=(10, 8))

    shap.summary_plot(
        loaded_shap_values_sw, 
        X_sw, 
        plot_type="dot", 
        show=False, 
        cmap=cmap,
        max_display=num_features
    )

    # 获取主图轴
    ax = plt.gca()
    ax.set_title("SHAP dot plot for water", fontsize=18)
    ax.tick_params(axis='both', labelsize=14)
    ax.xaxis.label.set_size(16)
    ax.yaxis.label.set_size(16)

    # =========================
    # 删除 shap 自带 colorbar
    # =========================
    fig = plt.gcf()
    if len(fig.axes) > 1:
        old_cbar_ax = fig.axes[-1]
        fig.delaxes(old_cbar_ax)

    # =========================
    # 手动创建 colorbar
    # =========================
    # 这里 [left, bottom, width, height] 可以自由调
    cbar_ax = fig.add_axes([0.88, 0.25, 0.03, 0.45])

    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.ax.tick_params(labelsize=16)

    # 设置 colorbar 标签
    cbar.set_label("Value", fontsize=16, rotation=90, labelpad=6)

    plt.savefig(path_part3_fig2 + 'shap_sw_summary_dot_plot.svg', bbox_inches='tight')
    plt.close()

else:
    print("Unable to create plots: SHAP values not loaded successfully.")

In [14]:
# top 5
import numpy as np
import matplotlib.colors as mcolors
# 获取特征重要性排序
feature_importance = np.abs(loaded_shap_values_sw.values).mean(0)
feature_importance_order = np.argsort(feature_importance)[::-1]

# 选择前五个最重要的特征
top_5_features = feature_importance_order[:5]

# 创建只包含前五个特征的新的SHAP值和特征数据
top_5_shap_values = loaded_shap_values_sw.values[:, top_5_features]
top_5_feature_names = X_sw.columns[top_5_features]
top_5_X_sw = X_sw.iloc[:, top_5_features]

# Summary plot (bar)
plt.figure(figsize=(8,4)) 
shap.summary_plot(top_5_shap_values, top_5_X_sw, plot_type="bar", show=False, color='#84aeb8', feature_names=top_5_feature_names)
# plt.title("SHAP Summary Bar Plot (Top 5 Features)")

# Add value labels
ax = plt.gca()
# for p in ax.patches:
#     width = p.get_width()
#     ax.text(width, p.get_y() + p.get_height()/2, f'{width:.3f}', 
#             ha='left', va='center')
ax.spines['top'].set_visible(True)
ax.spines['bottom'].set_visible(False)
plt.xlabel('Mean |SHAP value|')
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')  # 标签也移到上面
plt.tight_layout()
plt.subplots_adjust(left=0.15,right=0.75,bottom=0.15,top=0.85)
plt.savefig(path_part3_fig + 'fig4_sw_bar_top5.svg', bbox_inches='tight')
plt.close()

# Summary plot (dot)

# 定义颜色
# max_color = '#8bd0e3'
# min_color = mcolors.to_rgba(max_color, alpha=0.1)  # 10% 的 #8bd0e3
# colors = [min_color, max_color]

# Create a custom colormap from white to red
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt

# 定义三种颜色
colors = ['#4d8cbf', '#ffddb8', '#c3473b']  
n_bins = 100  # 渐变分段数
cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=n_bins)

plt.figure(figsize=(8, 6))
shap.summary_plot(
    top_5_shap_values,
    top_5_X_sw,
    plot_type="dot",
    show=False,
    cmap=cmap,
    feature_names=top_5_feature_names,
    alpha=0.1
)
plt.xlim(-1, 1)
# plt.title("SHAP Summary Dot Plot (Top 5 Features)")
plt.tight_layout()
plt.savefig(path_part3_fig + 'fig4_sw_dot_top5.svg', bbox_inches='tight')
plt.close()


#### lr

In [15]:
int_rdm = 202603
str_describe_lr = 'lr'
save_file = True

list_pfas = ['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 
             'PFOS', 'FOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', 
             'HFPO-DA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS']

In [16]:

df_lr_raw = pd.read_csv(path_part0_match + "lr_refcv_match.csv")

print(df_lr_raw.shape)
df_lr_data_raw = mf.select_data(df_lr_raw, path_file, 'bio')
print(df_lr_data_raw.shape)
# 在lr_match基础上用已经建立的sw模型，补充所有的sw_value数据
print(df_lr_data_raw['posname'].unique().tolist())
print(len(df_lr_data_raw['posname'].unique().tolist()))
df_lr_data_raw['value'], lam = boxcox(df_lr_data_raw['value'])
print(f'Box-Cox Lambda: {lam}')

mean_val = df_lr_data_raw['value'].mean()
std_val = df_lr_data_raw['value'].std(ddof=0)
print(f"[ZScore] Mean: {mean_val}, Std: {std_val}")
df_lr_data_raw['value'] = (df_lr_data_raw['value'] - mean_val) / std_val
list_inv_parm = [mean_val, std_val, lam]


df_lr_data_raw['year'] = (df_lr_data_raw['year'] - 2000) / (2020 - 2000)

df_lr_train = df_lr_data_raw[df_lr_data_raw['posname'].isin(list_pfas)]
df_lr_tranfer = df_lr_data_raw[~df_lr_data_raw['posname'].isin(list_pfas)]

columns_to_drop = ['lon_grid', 'lat_grid', 'posname']
df_lr_data = df_lr_train.drop(columns=columns_to_drop)

print(df_lr_data.columns)
print(df_lr_data.shape)
# value列范围
print(f"Value range: {df_lr_data['value'].min()} to {df_lr_data['value'].max()}")



(21550, 53)
(21550, 49)
['PFBS', 'PFDA', 'PFDS', 'PFDoDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFOS', 'PFTeDA', 'PFTrDA', 'PFUnDA', 'PFHpS', 'PFNA', 'PFOA', 'PFPeA', 'FOSA', 'PFPeDA', 'PFBA', 'PFHxDA', 'PFODA', '6:2 Cl-PFESA', 'EtFOSAA', '4:2 FTSA', '6:2 FTSA', '8:2 FTSA', 'EtFOSA', 'HFPO-DA', 'MeFOSA', 'MeFOSAA', 'PFNS', 'PFPeS', 'EtFOSE', '6:8 PFPIA', '6:2 diPAP', '8:2 diPAP', 'FOSAA', '6:2/8:2 diPAP', '10:2 FTSA']
38
Box-Cox Lambda: -0.10693903224929828
[ZScore] Mean: -2.027964285321613, Std: 2.2294033230935657
Index(['z', 'ts', 'sp_troph', 'manufacturing', 'sfcWind', 'organ_muscle',
       'mrro', 'ta', 'sw_value', 'solubility', 'clt', 'ps', 'clothing',
       'log_Koil_w', 'GDP', 'huss', 'po_f_carbon', 'potential_contamination',
       'sp_length', 'log_pKa', 'value', 'organ_liver', 'prsn', 'po_carbon',
       'fluorite_consumption', 'wrap_consumption', 'log_D7_4', 'evspsbl',
       'po_m_w', 'SWD_LDF_CH4', 'distance_to_sources', 'urban', 'WWT_CH4',
       'TOTALS_CO2', 'zg', 'paper_consu

In [18]:


selected_features_lr = pd.read_csv(path_part3_lr + str_describe_lr + "_rfecv_features_LGBMcv.csv")
selected_features_lr = selected_features_lr[selected_features_lr["Rank"] == 1]["Feature"].values

best_params = pd.read_csv(path_part3_lr + 'ml_cv_best.csv')

model_params = best_params[best_params["model"] == 'LGBM'].iloc[0]
if pd.isna(model_params["max_depth"]) or str(model_params["max_depth"]).lower() == 'none':
    param_max_depth = None
else:
    param_max_depth = int(float(model_params["max_depth"]))

model_fish = LGBMRegressor(
    max_depth=param_max_depth,
    learning_rate=model_params["learning_rate"],
    min_child_samples=int(model_params["min_child_samples"]),
    num_leaves=int(model_params["num_leaves"]),
    n_estimators=int(model_params["n_estimators"]),
    random_state=20260300,
    subsample=0.8,
    subsample_freq=1,
    n_jobs=12
)


X = df_lr_data[selected_features_lr]
y = df_lr_data['value']


In [19]:
# 一次性计算shap
import shap
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 确保模型已经训练
model_fish.fit(X, y)

# 创建 explainer 并计算 SHAP 值
try:
    explainer_fish = shap.TreeExplainer(model_fish)
    shap_values_fish = explainer_fish(X)
    
    # 保存 SHAP 值到本地文件
    shap_save_path_fish = path_part3_fig + 'shap_values_lr.pkl'
    with open(shap_save_path_fish, 'wb') as f:
        pickle.dump(shap_values_fish, f)
    
    print("SHAP values calculated and saved successfully.")
except Exception as e:
    print(f"Error calculating or saving SHAP values: {e}")



SHAP values calculated and saved successfully.


In [28]:
path_part3_fig3 = 'F:/User_file/wyy/SPDB/part3_forecast/lr/fig/'

In [30]:
# 所有变量
import shap
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
shap_save_path_fish = path_part3_fig + 'shap_values_lr.pkl'
try:
    with open(shap_save_path_fish, 'rb') as f:
        loaded_shap_values_fish = pickle.load(f)
    print("SHAP values loaded successfully.")
except Exception as e:
    print(f"Error loading SHAP values: {e}")
    loaded_shap_values_fish = None
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

if loaded_shap_values_fish is not None:
    # ===== 全局字体设置 =====
    plt.rcParams.update({
        'font.size': 16,         # 默认字体大小
        'axes.titlesize': 18,    # 标题字体
        'axes.labelsize': 16,    # 坐标轴标签字体
        'xtick.labelsize': 14,   # x轴刻度字体
        'ytick.labelsize': 14,   # y轴刻度字体
        'legend.fontsize': 14    # 图例字体
    })

    # 计算总变量数量
    # X.shape[1]
    num_features = 20

    # =========================
    # Summary plot (bar)
    # =========================
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        loaded_shap_values_fish,
        X,
        plot_type="bar",
        show=False,
        color='#7d84a8',
        max_display=num_features
    )

    ax = plt.gca()
    ax.set_title("SHAP Summary Bar Plot", fontsize=18)

    # x轴label分成两行
    ax.set_xlabel("Mean |SHAP value|\n(average impact on model output magnitude)", fontsize=14)

    # 调整坐标轴刻度字体
    ax.tick_params(axis='both', labelsize=14)

    # 添加数值标签，并放大字体
    for p in ax.patches:
        width = p.get_width()
        ax.text(
            width,
            p.get_y() + p.get_height() / 2,
            f'{width:.3f}',
            ha='left',
            va='center',
            fontsize=13
        )

    plt.tight_layout()
    plt.savefig(path_part3_fig3 + 'shap_lr_summary_bar_plot.svg', bbox_inches='tight')
    plt.close()

    # =========================
    # Summary plot (dot)
    # =========================
    colors = ['#FFFF00', '#FF0000']
    n_bins = 100
    cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=n_bins)

    fig = plt.figure(figsize=(10, 8))

    shap.summary_plot(
        loaded_shap_values_fish,
        X,
        plot_type="dot",
        show=False,
        cmap=cmap,
        max_display=num_features
    )

    # 获取主图轴
    ax = plt.gca()
    ax.set_title("SHAP Summary Dot Plot", fontsize=18)
    ax.tick_params(axis='both', labelsize=14)
    ax.xaxis.label.set_size(16)
    ax.yaxis.label.set_size(16)

    # =========================
    # 删除 shap 自带 colorbar
    # =========================
    fig = plt.gcf()
    if len(fig.axes) > 1:
        old_cbar_ax = fig.axes[-1]
        fig.delaxes(old_cbar_ax)

    # =========================
    # 手动创建 colorbar
    # =========================
    cbar_ax = fig.add_axes([0.88, 0.25, 0.03, 0.45])

    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.ax.tick_params(labelsize=16)

    # 设置 colorbar 标签
    cbar.set_label("Value", fontsize=16, rotation=90, labelpad=6)

    plt.savefig(path_part3_fig3 + 'shap_lr_summary_dot_plot.svg', bbox_inches='tight')
    plt.close()

else:
    print("Unable to create plots: SHAP values not loaded successfully.")

SHAP values loaded successfully.


In [13]:
# top 5
import numpy as np
import matplotlib.colors as mcolors
# 获取特征重要性排序
feature_importance = np.abs(loaded_shap_values_fish.values).mean(0)
feature_importance_order = np.argsort(feature_importance)[::-1]

# 选择前五个最重要的特征
top_5_features = feature_importance_order[:6]

# 创建只包含前五个特征的新的SHAP值和特征数据
top_5_shap_values = loaded_shap_values_fish.values[:, top_5_features]
top_5_feature_names = X.columns[top_5_features]
top_5_X = X.iloc[:, top_5_features]

# Summary plot (bar)
shap.summary_plot(top_5_shap_values, top_5_X, plot_type="bar", show=False, color='#7d84a8', feature_names=top_5_feature_names)
plt.title("SHAP Summary Bar Plot (Top 5 Features)")

# Add value labels
ax = plt.gca()
for p in ax.patches:
    width = p.get_width()
    ax.text(width, p.get_y() + p.get_height()/2, f'{width:.3f}', 
            ha='left', va='center')

plt.tight_layout()
plt.savefig(path_part3_fig + 'fig4_lr_bar_top6.svg', bbox_inches='tight')
plt.close()

# Summary plot (dot)

# 定义颜色
# max_color = '#7d84a8'
# min_color = mcolors.to_rgba(max_color, alpha=0.2)  # 10% 的 #8bd0e3
# colors = [min_color, max_color]

# Create a custom colormap from white to red
# #FFFFFF  白色 #FFFF00  黄色
colors = ['#FFFF00', '#FF0000']
n_bins = 100
cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=n_bins)

shap.summary_plot(top_5_shap_values, top_5_X, plot_type="dot", show=False, cmap=cmap, feature_names=top_5_feature_names, alpha=0.1, max_display=6)
plt.title("SHAP Summary Dot Plot (Top 5 Features)")
plt.tight_layout()
plt.savefig(path_part3_fig + 'fig4_lr_dot_top6.svg', bbox_inches='tight')
plt.close()